# 시계열 ACF 연습 (PyTorch)

statsmodels 결과와 PyTorch 직접 구현을 비교합니다.

**GitHub(특히 아이패드) 미리보기:** LaTeX 수식이 있으면 `Unable to render code block`이 날 수 있어, 이 노트북은 수식 없이 작성했습니다. 로컬 Jupyter에서 Run All 하세요.

In [ ]:
from statsmodels.tsa.stattools import acf as sm_acf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import warnings
warnings.filterwarnings('ignore')

# %matplotlib inline  # 로컬 Jupyter에서 실행할 때 주석 해제


## 1. ACF 구현

표본 자기상관(sample ACF):

```text
rho_hat(k) = sum_{t>=k} (x_t - mean) * (x_{t-k} - mean) / sum_t (x_t - mean)^2
rho_hat(0) = 1
```

**구현 아이디어** (conv1d / nn 없이, for-loop 없이)
- k, t 격자를 unsqueeze로 브로드캐스팅
- lag k에서 t >= k 인 시점만 xc[t] * xc[t-k] 합산
- 사용 연산: mean, mul, sum, 인덱싱, torch.arange

In [ ]:
def acf(x: torch.Tensor, max_lag: int) -> torch.Tensor:
    """
    표본 ACF. 텐서 인덱싱·브로드캐스팅만 사용.
    x: (n,) 1차원 시계열 → (max_lag+1,)
    """
    x = x.float().flatten()
    xc = x - x.mean()
    n = xc.numel()
    denom = (xc * xc).sum()

    k = torch.arange(max_lag + 1, device=x.device).unsqueeze(1)
    t = torch.arange(n, device=x.device).unsqueeze(0)
    valid = t >= k

    prod = xc[t] * xc[(t - k).clamp(min=0)]
    acov = (prod * valid).sum(dim=1)

    return acov / denom


## 2. 데이터 로드

In [ ]:
df = pd.read_csv('../data/widget_sales.csv')
df.head()


## 3. 원시 시계열 시각화

In [ ]:
fig, ax = plt.subplots()

ax.plot(df['widget_sales'])
ax.set_xlabel('Time')
ax.set_ylabel('Widget sales (k$)')

plt.xticks(
    [0, 30, 57, 87, 116, 145, 175, 204, 234, 264, 293, 323, 352, 382, 409, 439, 468, 498],
    ['Jan 2019', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug',
     'Sep', 'Oct', 'Nov', 'Dec', 'Jan 2020', 'Feb', 'Mar', 'Apr', 'May', 'Jun'])

fig.autofmt_xdate()
plt.tight_layout()


## 4. 원시 시계열 ACF - PyTorch vs statsmodels

In [ ]:
MAX_LAG = 20

series = df['widget_sales'].to_numpy()
x = torch.tensor(series, dtype=torch.float32)

acf_pt = acf(x, MAX_LAG)
acf_sm = torch.tensor(sm_acf(series, nlags=MAX_LAG, fft=True), dtype=torch.float32)

print('pytorch vs statsmodels:', torch.allclose(acf_pt, acf_sm, atol=1e-5))
print(torch.stack([acf_pt, acf_sm], dim=1), '\n(columns: pytorch, statsmodels)')


## 5. 1차 차분 시계열

비정상 시계열은 차분 후 ACF 패턴이 달라집니다.

In [ ]:
widget_sales_diff = np.diff(df['widget_sales'].to_numpy(), n=1)

### 차분 시계열 시각화

In [ ]:
fig, ax = plt.subplots()

ax.plot(widget_sales_diff)
ax.set_xlabel('Time')
ax.set_ylabel('Widget sales - diff (k$)')

n = len(widget_sales_diff)
tick_idx = [0, 30, 57, 87, 116, 145, 175, 204, 234, 264, 293, 323, 352, 382, 409, 439, 468, n - 1]
tick_idx = [i for i in tick_idx if i < n]
tick_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug',
               'Sep', 'Oct', 'Nov', 'Dec', '2020', 'Feb', 'Mar', 'Apr', 'May', 'Jun'][:len(tick_idx)]
plt.xticks(tick_idx, tick_labels)

fig.autofmt_xdate()
plt.tight_layout()


## 6. 차분 시계열 ACF - PyTorch vs statsmodels

In [ ]:
MAX_LAG = 20

series = widget_sales_diff  # numpy ndarray (차분 결과)
x = torch.tensor(series, dtype=torch.float32)

acf_pt = acf(x, MAX_LAG)
acf_sm = torch.tensor(sm_acf(series, nlags=MAX_LAG, fft=True), dtype=torch.float32)

print('pytorch vs statsmodels:', torch.allclose(acf_pt, acf_sm, atol=1e-5))
print(torch.stack([acf_pt, acf_sm], dim=1), '\n(columns: pytorch, statsmodels)')
